Create deterministic pseudo_UIDs, compute rolling/expanding transaction aggregates, extract cyclical time features from TransactionDT, and record interaction flags.  
No target encoding. Only feature mappings that do not use isFraud as an aggregate source.  
Input: data/interim/pruned_eda.parquet  
Output: data/processed/features_static.parquet

In [1]:
import os
import numpy as np
import pandas as pd
from src.utils import reduce_mem_usage
from src.data.preprocessing import rolling_24h_stats_per_user
from src.config import interim_data_path, processed_data_path, artifacts_path
import json

In [2]:
interim_data_file = os.path.join( interim_data_path, "pruned_eda.parquet")
os.makedirs(processed_data_path, exist_ok=True)

df = pd.read_parquet(interim_data_file)
df = reduce_mem_usage(df)

Memory usage decreased to 774.48 MB (0.1% reduction)


In [3]:
# Create rolling windows for last 24h, for amount of transactiones per UID 
# IEEE-CIS anonymized user identity. Real systems track a user_id, we synthesize one using immutable or slowly-changing attributes
df['card_birthday'] = df['TransactionDTDays'] - df['D1']   # D1 is days after card registration, so we try to guess approx creation date of a card
df['Pseudo_UID'] = df['card1'].astype('str')+'-'+df['card2'].astype('str') +'-'+df['card3'].astype('str') +'-'+df['card4'].astype('str') +'-'+df['card5'].astype('str') +'-'+df['card6'].astype('str')+'-'+df['addr1'].astype('str') +'-'+df['addr2'].astype('str') + df['card_birthday'].astype('str')


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_6256\3508039662.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['card_birthday'] = df['TransactionDTDays'] - df['D1']   # D1 is days after card registration, so we try to guess approx creation date of a card
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_6256\3508039662.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Pseudo_UID'] = df['card1'].astype('str')+'-'+df['card2'].astype('str') +'-'+df['card3'].astype('str') +'-'+df['card4'].astype('str') +'-'+df['card5'].astype('s

In [4]:
df['hour_of_day']=(df['TransactionDT'] // 3600) % 24
df['day_of_week']=(df['TransactionDT'] // (3600*24)) % 7

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_6256\2182029119.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['hour_of_day']=(df['TransactionDT'] // 3600) % 24
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_6256\2182029119.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['day_of_week']=(df['TransactionDT'] // (3600*24)) % 7


This script calculates rolling 24-hour historical transaction statistics (count and average amount) per user without data leakage (it looks strictly at past transactions, excluding the current one).Instead of using slow pandas operations like df.groupby(...).rolling('24h'), it converts the data into raw 1D NumPy arrays and uses binary search (np.searchsorted) and prefix sums (np.cumsum) to compute these rolling metrics at $O(N \log K)$ speed with minimal memory overhead.

In [5]:
df=rolling_24h_stats_per_user(df)

C:\Users\ADMIN\Documents\DataScience\ieee-fraud-detection\src\data\preprocessing.py:65: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['uid_count_24h'] = counts     # count of transactions by the user in prior 24 h
C:\Users\ADMIN\Documents\DataScience\ieee-fraud-detection\src\data\preprocessing.py:66: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['uid_amt_mean_24h'] = np.where(np.isnan(means), amts, means)  # average transaction amount spent by the user in prior 24 h


In [6]:
# Time elapsed since previous transaction for this UID
df['uid_prev_dt'] = df.groupby('Pseudo_UID')['datetime'].shift(1)
df['uid_dt_diff'] = df['datetime'] - df['uid_prev_dt']  # NaNs here mean 1st transaction

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_6256\3216060668.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['uid_prev_dt'] = df.groupby('Pseudo_UID')['datetime'].shift(1)
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_6256\3216060668.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['uid_dt_diff'] = df['datetime'] - df['uid_prev_dt']  # NaNs here mean 1st transaction


In [7]:
# drop columns TransactionID, datetime, uid_prev_dt
df.drop(columns=['TransactionID', 'uid_prev_dt', 'datetime'], inplace=True)

In [8]:
# Frequency Encoding for High-Cardinality Categoricals
freq_cols = [
    # Card details
    'card1', 'card2', 'card3', 'card5',
    
    # Location
    'addr1', 'addr2',
    
    # Email providers
    'P_emaildomain', 'R_emaildomain',
    
    # Device & Identity metadata
    'DeviceInfo', 'id_30', 'id_31', 'id_33',
    
    # Composite proxy
    'Pseudo_UID'
]

for col in freq_cols:
    if col in df.columns:
        freq = df[col].value_counts(dropna=True)
        df[f'{col}_freq'] = df[col].map(freq).astype(np.float32)

df = reduce_mem_usage(df)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_6256\2233324321.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_freq'] = df[col].map(freq).astype(np.float32)
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_6256\2233324321.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_freq'] = df[col].map(freq).astype(np.float32)
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_6256\2233324321.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has

Memory usage decreased to 837.60 MB (0.6% reduction)


In [11]:
# List of columns to convert to category type for LGBM
categorical_cols = [
    # Card & Location features
    'card1', 'card2', 'card3', 'card5', 
    'addr1', 'addr2',
    
    # Match flags (excluding target-encoded M4)
    'M1', 'M2', 'M3', 'M5', 'M6', 'M7', 'M8', 'M9',
    
    # Identity details
    'DeviceType', 'DeviceInfo',
    'id_12', 'id_13', 'id_14', 'id_15', 'id_16', 'id_17', 'id_18', 'id_19',
    'id_20', 'id_21', 'id_22', 'id_23', 'id_24', 'id_25', 'id_26', 'id_27',
    'id_28', 'id_29', 'id_30', 'id_31', 'id_32', 'id_33', 'id_34', 'id_35',
    'id_36', 'id_37', 'id_38',
    
    # Time groupings
    'hour_of_day', 'day_of_week'
]

# Convert only columns that are present in your feature matrix
for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].astype('category')

In [13]:
output_path = os.path.join(processed_data_path, "features_static.parquet")
df.to_parquet(output_path, index=False, engine="pyarrow")
print(f"Features successfully engineered and saved to {output_path}. Shape: {df.shape}")

Features successfully engineered and saved to C:\Users\ADMIN\Documents\DataScience\ieee-fraud-detection\data\processed\features_static.parquet. Shape: (590540, 321)
